"# Q-BitNet + LingBot-World v3 — Continuous Cloud Training\n",
    "\n",
    "This notebook trains your model **continuously** on Google Colab with:\n",
    "- **Auto-resume** from Google Drive checkpoints (survives disconnects)\n",
    "- **Periodic saves** every 200 steps + end-of-epoch saves\n",
    "- **Keep-alive** thread to prevent idle disconnects\n",
    "- **Crash recovery** — auto-retries on CUDA OOM or runtime errors\n",
    "- **Full state save** — model + optimizer + brain modules + epoch/step\n",
    "- **TPU support** — auto-detects TPU v3-8 and uses PyTorch/XLA\n",
    "\n",
    "## Setup\n",
    "1. **Runtime → Change runtime type** → choose **T4 GPU**, **A100**, or **TPU v2**\n",
    "2. Upload your `stark/` folder to Google Drive (or clone from GitHub)\n",
    "3. Run all cells in order\n",
    "\n",
    "## After a disconnect\n",
    "Just re-run all cells — it auto-resumes from the last checkpoint.\n",
    "\n",
    "## TPU notes\n",
    "- If using TPU, you MUST run the TPU install cell (Cell 3b) before training\n",
    "- TPU runs in bfloat16 natively (no GradScaler needed)\n",
    "- TPU is ~2-4x faster than T4 GPU"

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Get the project code

Choose **one** of the options below.

### Option A: Project already on Google Drive

In [ ]:
# Option A: stark/ is already in your Google Drive
import os
os.chdir('/content/drive/MyDrive/stark')
print(f'Working dir: {os.getcwd()}')
!ls -la

"## Cell 3b: Install TPU support (ONLY if using TPU runtime)\n",
    "\n",
    "Skip this cell if you're using GPU. Only run this if you selected **TPU** as your runtime type.\n",
    "\n",
    "The torch-xla version must match your torch version. Colab currently ships torch 2.5.0."

In [ ]:
"# ONLY run this if using TPU runtime!\n",
    "# This installs torch-xla which is required for TPU support.\n",
    "# Check your torch version first:\n",
    "import torch\n",
    "print(f'torch version: {torch.__version__}')\n",
    "\n",
    "# Install torch-xla matching your torch version.\n",
    "# For torch 2.5.0 (Colab default as of 2025):\n",
    "!pip install -q cloud-tpu-client torch-xla==2.5.0\n",
    "\n",
    "# Verify TPU is detected:\n",
    "import os\n",
    "if 'COLAB_TPU_ADDR' in os.environ:\n",
    "    import torch_xla.core.xla_model as xm\n",
    "    device = xm.xla_device()\n",
    "    print(f'TPU detected: {device}')\n",
    "else:\n",
    "    print('WARNING: No TPU detected! Make sure Runtime type is set to TPU.')\n",
    "    print('If you are using GPU, you can skip this cell.'"

### Option B: Clone from GitHub

In [ ]:
# Option B: Clone from GitHub (replace with your repo URL)
# !git clone https://github.com/YOUR_USERNAME/stark.git /content/stark
# import os
# os.chdir('/content/stark')
# print(f'Working dir: {os.getcwd()}')

"import torch\n",
    "import os\n",
    "\n",
    "# Check for TPU first\n",
    "if 'COLAB_TPU_ADDR' in os.environ:\n",
    "    try:\n",
    "        import torch_xla.core.xla_model as xm\n",
    "        device = xm.xla_device()\n",
    "        print(f'TPU detected: {device}')\n",
    "        print('Ready for TPU training!')\n",
    "    except ImportError:\n",
    "        print('TPU detected but torch_xla not installed!')\n",
    "        print('Run Cell 3b (TPU install) first!')\n",
    "elif torch.cuda.is_available():\n",
    "    print(f'GPU: {torch.cuda.get_device_name(0)}')\n",
    "    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')\n",
    "    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')\n",
    "    print('Ready for GPU training!')\n",
    "else:\n",
    "    print('WARNING: No GPU or TPU detected!')\n",
    "    print('Go to Runtime → Change runtime type → T4 GPU or TPU'"

In [ ]:
!pip install -q transformers>=4.45 datasets torch sentencepiece huggingface_hub accelerate

## Cell 4: Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
else:
    print('WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

## Cell 5: Start continuous training

This cell runs the training loop. It will:
1. Download the base model (~5 min, first time only)
2. Check for existing checkpoints on Drive and resume if found
3. Train continuously, saving checkpoints every 200 steps + every epoch
4. Auto-retry on crashes (OOM, runtime errors)

**If Colab disconnects**, just re-run all cells — it picks up from the last checkpoint.

In [ ]:
!python colab/colab_train.py

"## Cell 5: Start continuous training\n",
    "\n",
    "This cell runs the training loop. It will:\n",
    "1. Download the base model (~5 min, first time only)\n",
    "2. Auto-detect GPU or TPU and configure accordingly\n",
    "3. Check for existing checkpoints on Drive and resume if found\n",
    "4. Train continuously, saving checkpoints every 200 steps + every epoch\n",
    "5. Auto-retry on crashes (OOM, runtime errors)\n",
    "\n",
    "**If Colab disconnects**, just re-run all cells — it picks up from the last checkpoint.\n",
    "\n",
    "**TPU users**: Make sure you ran Cell 3b (TPU install) before this cell."

In [ ]:
import os, json
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/stark_checkpoints')
if ckpt_dir.exists():
    ckpts = sorted(ckpt_dir.glob('ckpt_step_*.pt'))
    print(f'Total checkpoints: {len(ckpts)}')
    if ckpts:
        latest = ckpts[-1]
        print(f'Latest: {latest.name}')
        # Load and show metadata
        import torch
        state = torch.load(latest, map_location='cpu', weights_only=False)
        print(f'  Epoch: {state["epoch"]}')
        print(f'  Step:  {state["global_step"]}')
        print(f'  Phi:   {state.get("phi", 0):.4f}')
        if 'metrics' in state:
            print(f'  Metrics: {state["metrics"]}')
else:
    print('No checkpoints directory found yet.')

## Cell 7 (optional): Download checkpoints to local machine

In [ ]:
# Zip all checkpoints and download
import shutil
from google.colab import files

shutil.make_archive('/content/stark_checkpoints', 'zip', 
                    '/content/drive/MyDrive/stark_checkpoints')
files.download('/content/stark_checkpoints.zip')

## Cell 8 (optional): Clean up all checkpoints (START FRESH)

**WARNING**: This deletes all saved checkpoints. Only run if you want to restart training from scratch.

In [ ]:
# WARNING: This deletes ALL checkpoints!
# Uncomment to run:
# import shutil
# shutil.rmtree('/content/drive/MyDrive/stark_checkpoints', ignore_errors=True)
# print('All checkpoints deleted. Next run will start fresh.')